
# 03 — Resultados de Scoring RADAR Cibest | Versión Deep Dive Interpretativa

**Fase ASUM-DM:** 4 — Modelado y lectura estratégica de resultados  
**Proyecto:** RADAR Cibest — Ranking de Atractivo y Diagnóstico Analítico Regional  
**Propósito:** convertir los resultados de TOPSIS, IPC, Trend y RADAR en una lectura ejecutiva orientada a decisión, conectando rankings, tesis de negocio, drivers, restricciones, atractivo común entre países, sensibilidad por línea y consecuencias estratégicas.

---

## Tesis central del notebook

RADAR no debe leerse como una tabla de posiciones. Debe leerse como un sistema de interpretación estratégica:

1. **TOPSIS** identifica atractivo estructural relativo: qué países se parecen más al ideal positivo definido por fundamentos macro, financieros, institucionales y digitales.
2. **IPC** mide afinidad específica con Colombia: qué mercados son más accesibles para Cibest por cercanía geográfica, cultural, migratoria y relacional.
3. **Trend** captura momentum macroeconómico reciente: qué países están mejorando en el margen.
4. **RADAR** integra los tres componentes para priorizar mercados bajo una tesis de internacionalización realista: atractivo absoluto + proximidad ejecutable + momentum.
5. **Rankings por línea** no son listas independientes: son variaciones de una misma matriz país, ponderada según la lógica económica de cada negocio.

La pregunta ejecutiva no es solo: **¿quién queda de primero?**  
La pregunta correcta es: **¿por qué queda arriba, qué tan separada es su ventaja, para qué línea de negocio es atractivo, qué drivers lo explican y qué riesgos de implementación persisten?**



## 1. Configuración reproducible y recarga controlada de módulos

Esta celda evita uno de los problemas más frecuentes del proyecto: ejecutar notebooks con módulos antiguos en memoria después de modificar `src/`. La recarga debe ocurrir **antes** de importar funciones sueltas.


In [262]:

# ---------------------------------------------------------------------------
# Configuración inicial, imports y recarga controlada de módulos
# ---------------------------------------------------------------------------
from __future__ import annotations

import sys
import re
import importlib
import inspect
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import HTML, display, Markdown

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

import src.utils as utils
import src.data_preparation.cleaning as cleaning
import src.data_preparation.feature_engineering as feature_engineering
import src.scoring.hybrid_scorer as hybrid_scorer
import src.scoring.ranking as ranking
import src.scoring.explainability as explainability
import src.scoring.monte_carlo as monte_carlo

importlib.invalidate_caches()

# Primero dependencias; luego módulos que importan esas dependencias.
importlib.reload(utils)
importlib.reload(cleaning)
importlib.reload(feature_engineering)
importlib.reload(ranking)
importlib.reload(explainability)
importlib.reload(monte_carlo)
importlib.reload(hybrid_scorer)

from src.utils import load_all_configs, setup_logger, resolve_data_path, get_variable_catalog
from src.data_preparation.cleaning import run_cleaning
from src.scoring.hybrid_scorer import prepare_decision_matrix, run_full_scoring, _build_business_line_weights
from src.scoring.explainability import (
    compute_all_business_line_contributions,
    compute_all_marginal_effects,
    combine_contribution_and_marginal,
    classify_driver_robustness,
    build_country_driver_table,
)

configs = load_all_configs()
setup_logger(configs["settings"].get("logging"))
variable_catalog = get_variable_catalog(configs["variables"])

print("cleaning file:", cleaning.__file__)
print("hybrid_scorer file:", hybrid_scorer.__file__)
print("run_cleaning line:", inspect.getsourcelines(cleaning.run_cleaning)[1])
print("prepare_decision_matrix line:", inspect.getsourcelines(hybrid_scorer.prepare_decision_matrix)[1])


cleaning file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\data_preparation\cleaning.py
hybrid_scorer file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\scoring\hybrid_scorer.py
run_cleaning line: 355
prepare_decision_matrix line: 37



## 2. Estilo visual Cibest y utilidades ejecutivas

Las visualizaciones privilegian lectura ejecutiva: poco ornamento, colores con significado, tablas con gradientes solo cuando ayudan a identificar patrones.


In [263]:

# ---------------------------------------------------------------------------
# Paleta Cibest y utilidades de visualización
# ---------------------------------------------------------------------------
CIBEST: Dict[str, str] = {
    "gray": "#2C2A28",
    "gray_light": "#CCCAC7",
    "yellow": "#FDD923",
    "gold": "#D6B302",
    "gold_light": "#FFF7D3",
    "gold_dark": "#8F7701",
    "gray_bg": "#F5F5F5",
    "gray_border": "#D0D0D0",
    "white": "#FFFFFF",
    "green": "#0BA682",
    "amber": "#FF7E41",
    "red": "#C62828",
}

TIER_COLORS: Dict[str, str] = {
    "Alta": CIBEST["green"],
    "Media-alta": CIBEST["gold"],
    "Media": CIBEST["amber"],
    "Baja": CIBEST["red"],
}


cmap_custom = LinearSegmentedColormap.from_list(
    "GrayToYellow",
    [CIBEST["gray_light"], CIBEST["yellow"]],
)

def style_table(
    df: pd.DataFrame,
    gradient_cols: Optional[List[str]] = None,
    gradient_cmap: str = "YlGnBu",
    format_dict: Optional[Dict[str, str]] = None,
):
    """Aplica estilo visual Cibest a un DataFrame."""
    styler = df.style.set_table_styles([
        {"selector": "th", "props": [
            ("background-color", CIBEST["gray"]),
            ("color", CIBEST["yellow"]),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "8px"),
            ("font-family", "Arial, sans-serif"),
        ]},
        {"selector": "td", "props": [
            ("padding", "6px 10px"),
            ("font-family", "Arial, sans-serif"),
            ("border-bottom", f"1px solid {CIBEST['gray_border']}"),
        ]},
    ])
    if gradient_cols:
        cols = [c for c in gradient_cols if c in df.columns]
        if cols:
            styler = styler.background_gradient(subset=cols, cmap=gradient_cmap)
    if format_dict:
        styler = styler.format(format_dict)
    return styler


def insight_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de insight."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['gold']}; background-color:{CIBEST['gold_light']};
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def risk_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de riesgo."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['red']}; background-color:#FDECEC;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def success_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de confirmación."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['green']}; background-color:#E8F7F3;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))

success_box("Entorno listo", "Estilo ejecutivo y módulos RADAR recargados correctamente.")



## 3. Carga del master y validación mínima

La lectura de resultados parte del `master_raw_YYYYMMDD.parquet` más reciente. Esta celda no interpreta todavía; valida que el insumo del scoring sea completo y trazable.


In [264]:

# ---------------------------------------------------------------------------
# Carga del master raw más reciente
# ---------------------------------------------------------------------------
raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])
pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

candidates = sorted(
    [path for path in raw_dir.glob("master_raw_*.parquet") if pattern.match(path.name)],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not candidates:
    raise FileNotFoundError("No se encontró master_raw_YYYYMMDD.parquet. Ejecute primero el notebook 01.")

master_path = candidates[0]
master = pd.read_parquet(master_path)

required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)
if missing_cols:
    raise ValueError(f"Master inválido. Faltan columnas requeridas: {sorted(missing_cols)}")

master = master.copy()
master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()
master["year"] = pd.to_numeric(master["year"], errors="coerce")
master["value"] = pd.to_numeric(master["value"], errors="coerce")

summary_master = pd.DataFrame({
    "métrica": ["archivo", "filas", "países", "variables", "fuentes", "año_min", "año_max", "incluye_gdp_growth"],
    "valor": [
        master_path.name,
        master.shape[0],
        master["country_iso3"].nunique(),
        master["variable"].nunique(),
        master["source"].nunique(),
        int(master["year"].min()),
        int(master["year"].max()),
        "Sí" if "gdp_growth" in master["variable"].unique() else "No",
    ],
})

display(style_table(summary_master))

if "gdp_growth" not in master["variable"].unique():
    risk_box("Advertencia crítica", "El master no contiene gdp_growth. El componente Trend no podrá calcularse correctamente.")


,métrica,valor
0,archivo,master_raw_20260625.parquet
1,filas,1288
2,países,30
3,variables,45
4,fuentes,3
5,año_min,1996
6,año_max,2026
7,incluye_gdp_growth,Sí



## 4. Preparación de matriz y política de exclusión por calidad

La matriz de scoring debe construirse con una regla conservadora: **primero se excluyen países con faltantes efectivos excesivos; luego se imputa solo lo residual**. Esto es relevante porque TOPSIS usa distancias al ideal positivo y negativo; países con demasiada imputación pueden alterar esos puntos de referencia.


In [265]:

# ---------------------------------------------------------------------------
# Preparación de matriz de decisión y exclusiones
# ---------------------------------------------------------------------------
wide_raw, decision_matrix, excluded_countries = prepare_decision_matrix(master, configs)

matrix_summary = pd.DataFrame({
    "métrica": [
        "wide_raw_shape",
        "decision_matrix_shape",
        "países_excluidos",
        "gdp_growth_en_wide_raw",
        "gdp_growth_en_decision_matrix",
    ],
    "valor": [
        str(wide_raw.shape),
        str(decision_matrix.shape),
        ", ".join(sorted(excluded_countries)) if excluded_countries else "Ninguno",
        "Sí" if "gdp_growth" in wide_raw.columns else "No",
        "Sí" if "gdp_growth" in decision_matrix.columns else "No",
    ],
})

display(style_table(matrix_summary))

if "gdp_growth" in decision_matrix.columns:
    raise ValueError("gdp_growth está entrando a TOPSIS. Debe usarse solo en Trend.")

if excluded_countries:
    risk_box(
        "Exclusiones por calidad antes de scoring",
        "Los países excluidos no entran a TOPSIS/RADAR. Esto evita que la imputación convierta países con baja cobertura en alternativas artificialmente comparables. "
        f"Excluidos: {', '.join(sorted(excluded_countries))}."
    )
else:
    success_box("Sin exclusiones por calidad", "Todos los países cumplen el umbral de cobertura efectiva definido en configuración.")


2026-06-25 21:30:04 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-25 21:30:04 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 | Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-25 21:30:04 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 | Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-25 21:30:04 | INFO     | src.data_preparation.cleaning:apply_freshness_filt

,métrica,valor
0,wide_raw_shape,"(25, 46)"
1,decision_matrix_shape,"(25, 35)"
2,países_excluidos,"BRB, CUB, GUY, HTI, VEN"
3,gdp_growth_en_wide_raw,Sí
4,gdp_growth_en_decision_matrix,No



## 5. Ejecución del scoring híbrido RADAR

La ejecución produce tres familias de resultados:

- **Global:** ranking estructural TOPSIS y ranking compuesto RADAR.
- **Componentes:** IPC y Trend, que explican movimientos respecto a TOPSIS.
- **Líneas de negocio:** rankings por IB, PF, AD, BD y CIB, cada uno con una tesis distinta de atractividad.


In [266]:

# ---------------------------------------------------------------------------
# Ejecución completa del scoring
# ---------------------------------------------------------------------------
results = run_full_scoring(master, configs, persist=True)

alpha = results["composite_weights"]["alpha"]
beta = results["composite_weights"]["beta"]
gamma = results["composite_weights"]["gamma"]

scoring_summary = pd.DataFrame({
    "métrica": ["alpha_TOPSIS", "beta_IPC", "gamma_Trend", "países_evaluados", "países_excluidos"],
    "valor": [
        alpha,
        beta,
        gamma,
        len(results["radar_global"]),
        ", ".join(sorted(results.get("excluded_countries", []))) if results.get("excluded_countries") else "Ninguno",
    ],
})

display(style_table(scoring_summary, format_dict={"valor": "{}"}))

insight_box(
    "Lectura del score compuesto",
    f"RADAR pondera {alpha:.0%} atractivo estructural TOPSIS, {beta:.0%} proximidad/afinidad con Colombia y {gamma:.0%} momentum macro reciente. "
    "Por diseño, el ranking final no replica exactamente el ranking estructural: mercados cercanos o con fuerte tendencia pueden subir, mientras mercados estructuralmente fuertes pero lejanos pueden bajar."
)


2026-06-25 21:30:04 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-25 21:30:04 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 | Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-25 21:30:04 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 | Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-25 21:30:04 | INFO     | src.data_preparation.cleaning:apply_freshness_filt

,métrica,valor
0,alpha_TOPSIS,0.6
1,beta_IPC,0.3
2,gamma_Trend,0.1
3,países_evaluados,24
4,países_excluidos,"BRB, CUB, GUY, HTI, VEN"



## 6. Ranking RADAR global: lectura ejecutiva del portafolio país

El ranking RADAR global debe interpretarse como una priorización inicial de mercados para discusión ejecutiva. No sustituye due diligence regulatoria, competitiva o financiera; identifica dónde tiene sentido profundizar.


In [267]:

# ---------------------------------------------------------------------------
# Ranking RADAR global y TOPSIS global
# ---------------------------------------------------------------------------
def ensure_country_column(df: pd.DataFrame) -> pd.DataFrame:
    """Asegura que country_iso3 exista como columna."""
    tmp = df.copy()
    if "country_iso3" not in tmp.columns:
        tmp = tmp.reset_index().rename(columns={"index": "country_iso3"})
    return tmp

radar_global = ensure_country_column(results["radar_global"]).sort_values("radar_score", ascending=False)
radar_global["radar_rank"] = radar_global["radar_score"].rank(ascending=False, method="min").astype(int)

global_topsis = ensure_country_column(results["global_ranking"]).sort_values("rank")

display(style_table(
    radar_global.head(15),
    gradient_cols=["radar_score"],
    gradient_cmap="RdYlGn",
    format_dict={"radar_score": "{:.3f}", "radar_rank": "{:.0f}"},
).set_caption("Top 15 — Ranking RADAR global"))

fig = px.bar(
    radar_global.head(15).sort_values("radar_score"),
    x="radar_score",
    y="country_iso3",
    orientation="h",
    color="radar_score",
    color_continuous_scale="RdYlGn",
    title="Top 15 mercados por score RADAR global",
)
fig.update_layout(xaxis_title="Score RADAR", yaxis_title="País")
fig.show()


,country_iso3,radar_score,rank,radar_rank
0,PAN,0.681,1,1
1,CRI,0.641,2,2
2,ESP,0.617,3,3
3,DOM,0.609,4,4
4,CHL,0.560,5,5
5,USA,0.541,6,6
6,ECU,0.527,7,7
7,URY,0.526,8,8
8,MEX,0.524,9,9
9,GTM,0.521,10,10


In [268]:

# ---------------------------------------------------------------------------
# Interpretación automática del Top RADAR
# ---------------------------------------------------------------------------
top_radar = radar_global.head(5)[["country_iso3", "radar_score", "radar_rank"]].copy()
leader = top_radar.iloc[0]

interpretation = f"""
### Lectura ejecutiva del Top RADAR

El país líder del ranking compuesto es **{leader['country_iso3']}**, con un score RADAR de **{leader['radar_score']:.3f}**.  
El Top 5 está compuesto por: **{', '.join(top_radar['country_iso3'].tolist())}**.

La lectura correcta no es que el primer país sea "universalmente mejor", sino que, bajo la combinación actual de atractivo estructural, proximidad con Colombia y momentum macro, presenta la mejor mezcla relativa. Si el gap frente al segundo país es bajo, la decisión debe comunicarse como una banda competitiva y no como una jerarquía estricta.
"""

display(Markdown(interpretation))



### Lectura ejecutiva del Top RADAR

El país líder del ranking compuesto es **PAN**, con un score RADAR de **0.681**.  
El Top 5 está compuesto por: **PAN, CRI, ESP, DOM, CHL**.

La lectura correcta no es que el primer país sea "universalmente mejor", sino que, bajo la combinación actual de atractivo estructural, proximidad con Colombia y momentum macro, presenta la mejor mezcla relativa. Si el gap frente al segundo país es bajo, la decisión debe comunicarse como una banda competitiva y no como una jerarquía estricta.



## 7. De TOPSIS a RADAR: quién sube, quién baja y por qué

Esta es una de las lecturas más importantes del notebook. TOPSIS responde **“qué país es estructuralmente atractivo”**; RADAR responde **“qué país es atractivo para Cibest, considerando además proximidad y momentum”**.


In [269]:

# ---------------------------------------------------------------------------
# Descomposición RADAR y movimientos frente a TOPSIS
# ---------------------------------------------------------------------------
ipc_df = ensure_country_column(results["ipc"])
trend_df = ensure_country_column(results["trend"])

topsis_series = global_topsis.set_index("country_iso3")["score"].rename("topsis_score")
ipc_series = ipc_df.set_index("country_iso3")["ipc"].rename("ipc")
trend_series = trend_df.set_index("country_iso3")["trend"].rename("trend")

component_df = pd.concat([topsis_series, ipc_series, trend_series], axis=1)
component_df["ipc"] = component_df["ipc"].fillna(component_df["ipc"].median())
component_df["trend"] = component_df["trend"].fillna(component_df["trend"].median())
component_df["aporte_topsis"] = alpha * component_df["topsis_score"]
component_df["aporte_ipc"] = beta * component_df["ipc"]
component_df["aporte_trend"] = gamma * component_df["trend"]
component_df["radar_score_calc"] = component_df[["aporte_topsis", "aporte_ipc", "aporte_trend"]].sum(axis=1)
component_df["rank_topsis"] = component_df["topsis_score"].rank(ascending=False, method="min").astype(int)
component_df["rank_radar"] = component_df["radar_score_calc"].rank(ascending=False, method="min").astype(int)
component_df["delta_rank_radar_vs_topsis"] = component_df["rank_topsis"] - component_df["rank_radar"]
component_display = component_df.sort_values("rank_radar").reset_index().rename(columns={"index": "country_iso3"})

display(style_table(
    component_display.head(20),
    gradient_cols=["topsis_score", "ipc", "trend", "radar_score_calc", "delta_rank_radar_vs_topsis"],
    gradient_cmap="RdYlGn",
    format_dict={
        "topsis_score": "{:.3f}",
        "ipc": "{:.3f}",
        "trend": "{:.3f}",
        "aporte_topsis": "{:.3f}",
        "aporte_ipc": "{:.3f}",
        "aporte_trend": "{:.3f}",
        "radar_score_calc": "{:.3f}",
        "rank_topsis": "{:.0f}",
        "rank_radar": "{:.0f}",
        "delta_rank_radar_vs_topsis": "{:+.0f}",
    },
).set_caption("Descomposición RADAR y movimiento frente a TOPSIS"))


,country_iso3,topsis_score,ipc,trend,aporte_topsis,aporte_ipc,aporte_trend,radar_score_calc,rank_topsis,rank_radar,delta_rank_radar_vs_topsis
0,PAN,0.497,0.942,1.000,0.298,0.283,0.100,0.681,9,1,+8
1,CRI,0.523,0.835,0.763,0.314,0.250,0.076,0.641,6,2,+4
2,ESP,0.627,0.596,0.621,0.376,0.179,0.062,0.617,3,3,+0
3,DOM,0.479,0.864,0.629,0.287,0.259,0.063,0.609,11,4,+7
4,CHL,0.592,0.668,0.037,0.355,0.200,0.004,0.560,4,5,-1
5,USA,0.685,0.341,0.278,0.411,0.102,0.028,0.541,1,6,-5
6,ECU,0.388,0.954,0.082,0.233,0.286,0.008,0.527,20,7,+13
7,URY,0.558,0.540,0.290,0.335,0.162,0.029,0.526,5,8,-3
8,MEX,0.471,0.703,0.303,0.283,0.211,0.030,0.524,12,9,+3
9,GTM,0.409,0.735,0.544,0.246,0.221,0.054,0.521,18,10,+8


In [270]:

# ---------------------------------------------------------------------------
# Lectura automática de ganadores y perdedores por IPC/Trend
# ---------------------------------------------------------------------------
raisers = component_display.sort_values("delta_rank_radar_vs_topsis", ascending=False).head(5)
fallers = component_display.sort_values("delta_rank_radar_vs_topsis", ascending=True).head(5)

text = f"""
### Movimientos estratégicos entre TOPSIS y RADAR

**Países que más suben al pasar de TOPSIS a RADAR:** {', '.join(raisers['country_iso3'].tolist())}.  
Estos mercados suelen beneficiarse de mayor IPC, mejor momentum o ambos. La implicación es que pueden no ser los más fuertes estructuralmente, pero sí ser más ejecutables para Cibest.

**Países que más bajan al pasar de TOPSIS a RADAR:** {', '.join(fallers['country_iso3'].tolist())}.  
Estos mercados suelen ser estructuralmente sólidos, pero pierden atractivo relativo cuando se incorpora cercanía con Colombia y tendencia reciente.

Esta comparación evita una lectura ingenua del ranking: un mercado puede ser excelente en abstracto, pero menos prioritario para Cibest si su distancia operativa o relacional es alta.
"""
display(Markdown(text))



### Movimientos estratégicos entre TOPSIS y RADAR

**Países que más suben al pasar de TOPSIS a RADAR:** ECU, PAN, GTM, HND, NIC.  
Estos mercados suelen beneficiarse de mayor IPC, mejor momentum o ambos. La implicación es que pueden no ser los más fuertes estructuralmente, pero sí ser más ejecutables para Cibest.

**Países que más bajan al pasar de TOPSIS a RADAR:** BRA, CAN, JAM, TTO, BHS.  
Estos mercados suelen ser estructuralmente sólidos, pero pierden atractivo relativo cuando se incorpora cercanía con Colombia y tendencia reciente.

Esta comparación evita una lectura ingenua del ranking: un mercado puede ser excelente en abstracto, pero menos prioritario para Cibest si su distancia operativa o relacional es alta.



## 8. Arquetipos estratégicos de países

Para pasar de ranking a decisión, clasificamos países según la combinación de estructura, proximidad y momentum. Esta clasificación ayuda a definir **qué tipo de conversación estratégica** corresponde a cada país.


In [271]:

# ---------------------------------------------------------------------------
# Arquetipos estratégicos por componentes
# ---------------------------------------------------------------------------
def classify_archetype(row: pd.Series) -> str:
    """Clasifica un país según su combinación relativa de TOPSIS, IPC y Trend."""
    high_topsis = row["topsis_score"] >= component_df["topsis_score"].quantile(0.70)
    high_ipc = row["ipc"] >= component_df["ipc"].quantile(0.70)
    high_trend = row["trend"] >= component_df["trend"].quantile(0.70)

    if high_topsis and high_ipc:
        return "Prioridad natural Cibest"
    if high_topsis and not high_ipc:
        return "Atractivo estructural lejano"
    if high_ipc and not high_topsis:
        return "Cercano pero requiere tesis selectiva"
    if high_trend and not high_topsis:
        return "Momentum táctico"
    return "Monitoreo / oportunidad condicionada"

archetypes = component_df.copy()
archetypes["archetype"] = archetypes.apply(classify_archetype, axis=1)
archetype_display = (
    archetypes.reset_index()
    .rename(columns={"index": "country_iso3"})
    .sort_values("radar_score_calc", ascending=False)
)

display(style_table(
    archetype_display[["country_iso3", "archetype", "topsis_score", "ipc", "trend", "radar_score_calc", "rank_radar"]].head(25),
    gradient_cols=["topsis_score", "ipc", "trend", "radar_score_calc"],
    gradient_cmap="RdYlGn",
    format_dict={"topsis_score": "{:.3f}", "ipc": "{:.3f}", "trend": "{:.3f}", "radar_score_calc": "{:.3f}", "rank_radar": "{:.0f}"},
).set_caption("Arquetipos estratégicos por combinación de componentes"))

archetype_counts = archetype_display["archetype"].value_counts().reset_index()
archetype_counts.columns = ["archetype", "n_countries"]
display(style_table(archetype_counts, gradient_cols=["n_countries"], gradient_cmap="RdYlGn_r", format_dict={"n_countries": "{:.0f}"}))


,country_iso3,archetype,topsis_score,ipc,trend,radar_score_calc,rank_radar
8,PAN,Cercano pero requiere tesis selectiva,0.497,0.942,1.000,0.681,1
5,CRI,Prioridad natural Cibest,0.523,0.835,0.763,0.641,2
2,ESP,Atractivo estructural lejano,0.627,0.596,0.621,0.617,3
10,DOM,Cercano pero requiere tesis selectiva,0.479,0.864,0.629,0.609,4
3,CHL,Atractivo estructural lejano,0.592,0.668,0.037,0.560,5
0,USA,Atractivo estructural lejano,0.685,0.341,0.278,0.541,6
19,ECU,Cercano pero requiere tesis selectiva,0.388,0.954,0.082,0.527,7
4,URY,Atractivo estructural lejano,0.558,0.540,0.290,0.526,8
11,MEX,Monitoreo / oportunidad condicionada,0.471,0.703,0.303,0.524,9
17,GTM,Monitoreo / oportunidad condicionada,0.409,0.735,0.544,0.521,10


,archetype,n_countries
0,Monitoreo / oportunidad condicionada,8
1,Cercano pero requiere tesis selectiva,6
2,Atractivo estructural lejano,6
3,Momentum táctico,3
4,Prioridad natural Cibest,1



## 9. Rankings por línea: cada línea expresa una tesis de atractividad distinta

Los rankings por línea no buscan ser completamente independientes. En servicios financieros, los países con mejor institucionalidad, digitalización y profundidad financiera tienden a ser atractivos para varias líneas. La pregunta relevante es si **las razones de atractividad** cambian entre líneas.


In [272]:
df_view = results["radar_by_line"].copy() #ver una línea específica.copy()

display(style_table(
    df_view[["country_iso3", "IB", "PF", "AD", "BD", "CIB", "GLOBAL", "rank_global"]].head(25),
    gradient_cols=["IB", "PF", "AD", "BD", "CIB", "GLOBAL"],
    gradient_cmap="RdYlGn",
    format_dict={
        "IB": "{:.3f}",
        "PF": "{:.3f}",
        "AD": "{:.3f}",
        "BD": "{:.3f}",
        "CIB": "{:.3f}",
        "GLOBAL": "{:.3f}",
    },
).set_caption("Score global por línea y general"))


,country_iso3,IB,PF,AD,BD,CIB,GLOBAL,rank_global
16,PAN,0.704,0.662,0.664,0.678,0.666,0.681,1
7,CRI,0.654,0.630,0.657,0.672,0.561,0.641,2
10,ESP,0.650,0.644,0.632,0.683,0.623,0.617,3
8,DOM,0.583,0.599,0.565,0.637,0.624,0.609,4
6,CHL,0.585,0.562,0.586,0.622,0.585,0.560,5
23,USA,0.554,0.506,0.610,0.603,0.550,0.541,6
9,ECU,0.551,0.520,0.434,0.566,0.552,0.527,7
22,URY,0.530,0.524,0.562,0.560,0.490,0.526,8
14,MEX,0.467,0.478,0.452,0.562,0.522,0.524,9
11,GTM,0.514,0.477,0.416,0.472,0.481,0.521,10


In [273]:

# ---------------------------------------------------------------------------
# Rankings TOPSIS por línea de negocio 
# ---------------------------------------------------------------------------
line_rankings: Dict[str, pd.DataFrame] = {}

for business_line, ranking_df in results["business_line_rankings"].items():
    tmp = ensure_country_column(ranking_df)
    tmp = tmp.sort_values("rank").copy()
    line_rankings[business_line] = tmp

    display(Markdown(f"### {business_line} — Top 12 países"))
    display(style_table(
        tmp.head(12),
        gradient_cols=["score"],
        gradient_cmap="RdYlGn",
        format_dict={"score": "{:.3f}", "rank": "{:.0f}"},
    ).set_caption(f"Ranking {business_line}"))


### IB — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.707,0.075696,0.182818,0.602323,0.690070,0.806634,0.755174,1,IB
1,ESP,0.683,0.083082,0.178818,0.569905,0.682145,0.730944,0.736835,2,IB
2,CAN,0.671,0.085484,0.174528,0.700119,0.585730,0.923344,0.668285,3,IB
3,CHL,0.635,0.093368,0.162349,0.572716,0.606922,0.740357,0.666526,4,IB
4,BHS,0.582,0.100167,0.139457,0.490480,0.585937,0.608889,0.549265,5,IB
5,BRA,0.569,0.110848,0.146111,0.525375,0.663634,0.362913,0.549839,6,IB
6,URY,0.566,0.106186,0.138396,0.531180,0.478461,0.774739,0.628212,7,IB
7,CRI,0.546,0.112899,0.135755,0.499425,0.501525,0.675389,0.542714,8,IB
8,PAN,0.536,0.115445,0.133622,0.508816,0.576563,0.447478,0.485130,9,IB
9,JAM,0.533,0.112009,0.127679,0.450975,0.563751,0.499410,0.435583,10,IB


### PF — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,ESP,0.673,0.107439,0.220660,0.601785,0.407268,0.716088,0.801321,1,PF
1,CAN,0.641,0.123699,0.220826,0.580706,0.381349,0.912720,0.743135,2,PF
2,USA,0.626,0.125320,0.209765,0.368215,0.384904,0.800436,0.772037,3,PF
3,CHL,0.597,0.127316,0.188435,0.557462,0.340902,0.739730,0.702810,4,PF
4,URY,0.555,0.134075,0.167108,0.449576,0.300913,0.772727,0.647786,5,PF
5,BRA,0.545,0.138403,0.166101,0.374663,0.368994,0.375422,0.640003,6,PF
6,ARG,0.531,0.141265,0.159985,0.240726,0.330764,0.401102,0.647275,7,PF
7,CRI,0.506,0.139396,0.142754,0.603669,0.332425,0.685725,0.543971,8,PF
8,JAM,0.489,0.146900,0.140852,0.531548,0.740282,0.515285,0.414473,9,PF
9,SUR,0.468,0.150123,0.131863,0.418761,0.466313,0.317696,0.478239,10,PF


### AD — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.800,0.064192,0.256010,0.596469,0.694537,0.863337,0.735109,1,AD
1,CAN,0.742,0.089534,0.258015,0.730481,0.655987,0.957619,0.559761,2,AD
2,ESP,0.652,0.109973,0.205836,0.588407,0.545833,0.737571,0.551622,3,AD
3,CHL,0.636,0.116551,0.204021,0.593937,0.576999,0.764572,0.483999,4,AD
4,URY,0.618,0.125091,0.202087,0.549772,0.457700,0.777732,0.418258,5,AD
5,CRI,0.551,0.143181,0.175847,0.513842,0.274312,0.698506,0.352634,6,AD
6,BHS,0.514,0.151948,0.160732,0.581412,0.564636,0.559446,0.432479,7,AD
7,BLZ,0.474,0.178236,0.160885,0.458552,0.319911,0.360680,0.602588,8,AD
8,PAN,0.470,0.162965,0.144406,0.542400,0.423607,0.428784,0.532170,9,AD
9,ARG,0.408,0.180109,0.124106,0.339557,0.365768,0.402441,0.421955,10,AD


### BD — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.789,0.054385,0.203082,0.812200,0.818230,0.797886,0.756028,1,BD
1,ESP,0.738,0.066911,0.188348,0.664895,0.754004,0.715501,0.784127,2,BD
2,CAN,0.718,0.073372,0.186385,0.713779,0.737819,0.911762,0.688799,3,BD
3,CHL,0.697,0.075573,0.173556,0.601183,0.704393,0.739814,0.759375,4,BD
4,BRA,0.645,0.089026,0.161960,0.714709,0.742134,0.375171,0.582621,5,BD
5,ARG,0.617,0.095377,0.153885,0.581802,0.592009,0.401832,0.705888,6,BD
6,URY,0.615,0.096691,0.154708,0.502265,0.570900,0.774467,0.724425,7,BD
7,CRI,0.575,0.102484,0.138650,0.474197,0.580007,0.686019,0.636604,8,BD
8,MEX,0.534,0.115590,0.132707,0.710308,0.405139,0.335169,0.510803,9,BD
9,BHS,0.526,0.125987,0.139740,0.366582,0.612948,0.600528,0.609106,10,BD


### CIB — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,CAN,0.725,0.075238,0.198379,0.658005,0.709912,0.940030,0.512759,1,CIB
1,USA,0.701,0.090022,0.210785,0.575674,0.766499,0.827973,0.683950,2,CIB
2,ESP,0.636,0.095232,0.166618,0.616682,0.617225,0.743458,0.513287,3,CIB
3,CHL,0.635,0.096917,0.168512,0.540104,0.650748,0.746662,0.459598,4,CIB
4,BHS,0.546,0.119169,0.143428,0.407939,0.614467,0.596803,0.412426,5,CIB
5,JAM,0.521,0.124948,0.135797,0.375096,0.616308,0.492826,0.281199,6,CIB
6,BRA,0.510,0.130374,0.135527,0.524212,0.547463,0.355436,0.395872,7,CIB
7,TTO,0.509,0.128756,0.133588,0.378126,0.616851,0.410912,0.260698,8,CIB
8,DOM,0.503,0.130228,0.131790,0.446482,0.536394,0.482710,0.283899,9,CIB
9,URY,0.498,0.130822,0.129935,0.394837,0.464082,0.769076,0.409129,10,CIB



## 10. Correlaciones entre rankings: atractivo común vs diferenciación de tesis

Una correlación alta entre líneas no es automáticamente un error. Puede significar dos cosas muy distintas:

- **Problema de configuración:** las líneas usan pesos demasiado parecidos.
- **Factor país común:** las líneas usan pesos distintos, pero los mismos países son buenos en varias dimensiones.

Por eso comparamos **correlación de rankings** y **similitud coseno entre vectores de pesos**.


In [274]:

# ---------------------------------------------------------------------------
# Auditoría de pesos por línea
# ---------------------------------------------------------------------------
def audit_business_line_weights(
    configs: Dict[str, Dict[str, Any]],
    decision_matrix: pd.DataFrame,
) -> pd.DataFrame:
    """Audita pesos efectivos usados por TOPSIS para cada línea de negocio."""
    business_lines = configs["business_lines"]["business_lines"]
    global_dim_weights = configs["weights"]["dimension_weights"]
    global_variable_weights = configs["weights"]["variable_weights"]

    rows: List[Dict[str, Any]] = []

    for bl_key, bl_cfg in business_lines.items():
        dim_weights_line, final_var_weights = _build_business_line_weights(
            business_line_cfg=bl_cfg,
            variable_weights_by_dim=global_variable_weights,
        )
        final_var_weights_filtered = {
            var: weight
            for var, weight in final_var_weights.items()
            if var in decision_matrix.columns
        }
        total_filtered = sum(final_var_weights_filtered.values())
        if total_filtered > 0:
            final_var_weights_filtered = {
                var: weight / total_filtered
                for var, weight in final_var_weights_filtered.items()
            }
        overrides = bl_cfg.get("variable_weight_overrides", {}) or {}

        for dim, vars_global in global_variable_weights.items():
            for var, global_var_weight in vars_global.items():
                override_weight = overrides.get(dim, {}).get(var)
                rows.append({
                    "business_line": bl_key,
                    "dimension": dim,
                    "variable": var,
                    "in_decision_matrix": var in decision_matrix.columns,
                    "global_dimension_weight": global_dim_weights.get(dim),
                    "line_dimension_weight": dim_weights_line.get(dim, 0.0),
                    "global_variable_weight_in_dim": global_var_weight,
                    "override_variable_weight_in_dim": override_weight,
                    "has_override": override_weight is not None,
                    "final_topsis_weight": final_var_weights_filtered.get(var, 0.0),
                })

    return pd.DataFrame(rows)

weights_audit = audit_business_line_weights(configs, decision_matrix)

weight_sum_check = (
    weights_audit[weights_audit["in_decision_matrix"]]
    .groupby("business_line")["final_topsis_weight"]
    .sum()
    .reset_index()
)

display(style_table(weight_sum_check, format_dict={"final_topsis_weight": "{:.6f}"}).set_caption("Validación: pesos finales por línea suman 1"))

missing_weighted = weights_audit[(~weights_audit["in_decision_matrix"]) & (weights_audit["final_topsis_weight"] > 0)]
if not missing_weighted.empty:
    risk_box("Variables ponderadas ausentes", "Hay variables ponderadas que no están en decision_matrix. Revisar configuración o extracción.")
    display(missing_weighted)
else:
    success_box("Auditoría de pesos", "No hay variables ponderadas ausentes en la matriz de decisión.")


,business_line,final_topsis_weight
0,AD,1.000000
1,BD,1.000000
2,CIB,1.000000
3,IB,1.000000
4,PF,1.000000


In [275]:

# ---------------------------------------------------------------------------
# Correlaciones de rankings y similitud de pesos
# ---------------------------------------------------------------------------
rank_vectors = {}
for line, df in line_rankings.items():
    rank_vectors[line] = df.set_index("country_iso3")["rank"]
rank_df = pd.DataFrame(rank_vectors)
rank_corr = rank_df.corr(method="spearman")

display(style_table(
    rank_corr,
    gradient_cols=rank_corr.columns.tolist(),
    gradient_cmap="YlOrRd",
    format_dict={col: "{:.3f}" for col in rank_corr.columns},
).set_caption("Correlación Spearman entre rankings por línea"))

weight_vectors = (
    weights_audit[weights_audit["in_decision_matrix"]]
    .pivot_table(
        index="business_line",
        columns="variable",
        values="final_topsis_weight",
        aggfunc="sum",
        fill_value=0.0,
    )
)

weight_cosine = pd.DataFrame(
    cosine_similarity(weight_vectors),
    index=weight_vectors.index,
    columns=weight_vectors.index,
)

display(style_table(
    weight_cosine,
    gradient_cols=weight_cosine.columns.tolist(),
    gradient_cmap="YlOrRd",
    format_dict={col: "{:.3f}" for col in weight_cosine.columns},
).set_caption("Similitud coseno entre vectores de pesos por línea"))


,IB,PF,AD,BD,CIB
IB,1.000,0.704,0.765,0.651,0.795
PF,0.704,1.000,0.863,0.883,0.652
AD,0.765,0.863,1.000,0.834,0.710
BD,0.651,0.883,0.834,1.000,0.737
CIB,0.795,0.652,0.710,0.737,1.000


business_line,AD,BD,CIB,IB,PF
business_line,,,,,
AD,1.000,0.490,0.500,0.564,0.370
BD,0.490,1.000,0.409,0.610,0.601
CIB,0.500,0.409,1.000,0.720,0.285
IB,0.564,0.610,0.720,1.000,0.365
PF,0.370,0.601,0.285,0.365,1.000


**Lectura ejecutiva.** Una correlación alta no implica error. Puede indicar que dos líneas comparten fundamentos país. La meta no es forzar rankings independientes, sino que cada línea responda a una tesis de negocio defendible.

---

Interpretación:
- Alta similitud de pesos + alta correlación de ranking:
    el problema está en configuración.

- Baja similitud de pesos + alta correlación de ranking:
    los países tienen estructura subyacente común.

In [276]:

# ---------------------------------------------------------------------------
# Lectura automática de correlación vs similitud de pesos
# ---------------------------------------------------------------------------
def pairwise_interpretation(rank_corr: pd.DataFrame, weight_cosine: pd.DataFrame) -> pd.DataFrame:
    """Cruza correlación de rankings y similitud de pesos para interpretar pares de líneas."""
    rows: List[Dict[str, Any]] = []
    lines = sorted(rank_corr.columns)

    for i, line_a in enumerate(lines):
        for line_b in lines[i + 1:]:
            corr = rank_corr.loc[line_a, line_b]
            cosine = weight_cosine.loc[line_a, line_b]
            if corr >= 0.85 and cosine >= 0.70:
                diagnosis = "Riesgo de redundancia de configuración"
            elif corr >= 0.85 and cosine < 0.70:
                diagnosis = "Factor país común con tesis diferenciadas"
            elif corr < 0.70 and cosine < 0.50:
                diagnosis = "Tesis claramente diferenciadas"
            else:
                diagnosis = "Relación moderada / esperada"

            rows.append({
                "line_a": line_a,
                "line_b": line_b,
                "rank_corr": corr,
                "weight_cosine": cosine,
                "diagnosis": diagnosis,
            })

    return pd.DataFrame(rows).sort_values("rank_corr", ascending=False)

pair_diagnosis = pairwise_interpretation(rank_corr, weight_cosine)

display(style_table(
    pair_diagnosis,
    gradient_cols=["rank_corr", "weight_cosine"],
    gradient_cmap="YlOrRd",
    format_dict={"rank_corr": "{:.3f}", "weight_cosine": "{:.3f}"},
).set_caption("Diagnóstico: correlación de ranking vs similitud de pesos"))

common_factor_pairs = pair_diagnosis.query("diagnosis == 'Factor país común con tesis diferenciadas'")
if not common_factor_pairs.empty:
    insight_box(
        "Atractivo común entre países",
        "Algunos rankings correlacionan alto aunque sus pesos sean distintos. Esto indica que ciertos países concentran atributos transversales —institucionalidad, digitalización, profundidad financiera y estabilidad— que los hacen atractivos para varias líneas a la vez."
    )


,line_a,line_b,rank_corr,weight_cosine,diagnosis
6,BD,PF,0.883,0.601,Factor país común con tesis diferenciadas
3,AD,PF,0.863,0.370,Factor país común con tesis diferenciadas
0,AD,BD,0.834,0.490,Relación moderada / esperada
7,CIB,IB,0.795,0.720,Relación moderada / esperada
2,AD,IB,0.765,0.564,Relación moderada / esperada
4,BD,CIB,0.737,0.409,Relación moderada / esperada
1,AD,CIB,0.710,0.500,Relación moderada / esperada
9,IB,PF,0.704,0.365,Relación moderada / esperada
8,CIB,PF,0.652,0.285,Tesis claramente diferenciadas
5,BD,IB,0.651,0.610,Relación moderada / esperada



## 11. Variables que realmente diferencian las tesis

La diferenciación de tesis no se evalúa solo con pesos por dimensión. Debe observarse qué variables tienen mayor dispersión de peso entre líneas. Estas variables son los verdaderos “separadores estratégicos” del modelo.


In [277]:

# ---------------------------------------------------------------------------
# Spread de pesos entre líneas
# ---------------------------------------------------------------------------
weight_spread = weight_vectors.T.copy()
weight_spread["max_weight"] = weight_spread.max(axis=1)
weight_spread["min_weight"] = weight_spread.min(axis=1)
weight_spread["spread"] = weight_spread["max_weight"] - weight_spread["min_weight"]
weight_spread["dominant_line"] = weight_vectors.idxmax(axis=0)
weight_spread = weight_spread.sort_values("spread", ascending=False).reset_index().rename(columns={"index": "variable"})

display(style_table(
    weight_spread.head(20),
    gradient_cols=["spread", "max_weight"],
    gradient_cmap="YlOrRd",
    format_dict={col: "{:.4f}" for col in weight_vectors.index.tolist() + ["max_weight", "min_weight", "spread"] if col in weight_spread.columns},
).set_caption("Variables que más diferencian las tesis de negocio"))

fig = px.bar(
    weight_spread.head(15).sort_values("spread"),
    x="spread",
    y="variable",
    orientation="h",
    color="dominant_line",
    title="Variables con mayor dispersión de peso entre líneas",
)
fig.update_layout(xaxis_title="Spread de peso", yaxis_title="Variable")
fig.show()


business_line,variable,AD,BD,CIB,IB,PF,max_weight,min_weight,spread,dominant_line
0,digital_payment_adults_pct,0.0264,0.0540,0.0020,0.0080,0.1880,0.1880,0.0020,0.1860,PF
1,stock_market_cap_gdp,0.0360,0.0025,0.1440,0.0135,0.0075,0.1440,0.0025,0.1415,CIB
2,regulatory_quality,0.1632,0.0330,0.0552,0.0630,0.0276,0.1632,0.0276,0.1356,AD
3,secure_internet_servers_per_million,0.1320,0.0480,0.0060,0.0120,0.0235,0.1320,0.0060,0.1260,AD
4,rule_of_law,0.1248,0.0270,0.0624,0.0690,0.0216,0.1248,0.0216,0.1032,AD
5,internet_users_pct,0.0462,0.1020,0.0034,0.0110,0.0517,0.1020,0.0034,0.0986,BD
6,personal_remittances_gdp,0.0072,0.0025,0.0160,0.0045,0.0950,0.0950,0.0025,0.0925,PF
7,domestic_credit_private_gdp,0.0132,0.0350,0.0640,0.0990,0.0175,0.0990,0.0132,0.0858,IB
8,account_ownership,0.0060,0.0900,0.0080,0.0540,0.0550,0.0900,0.0060,0.0840,BD
9,mobile_subscriptions,0.0231,0.0660,0.0016,0.0090,0.0846,0.0846,0.0016,0.0830,PF



## 12. Top-N overlap: similitud decisional, no solo estadística

La correlación resume todo el ranking, pero la decisión ejecutiva suele concentrarse en Top 5 o Top 10. Dos líneas pueden correlacionar alto y aun así recomendar prioridades diferentes en el tramo relevante.


In [278]:

# ---------------------------------------------------------------------------
# Overlap Top-N entre líneas
# ---------------------------------------------------------------------------
def topn_overlap(rankings_by_line: Dict[str, pd.DataFrame], n: int = 5) -> pd.DataFrame:
    """Calcula solapamiento de Top-N entre rankings de líneas."""
    rows: List[Dict[str, Any]] = []
    lines = list(rankings_by_line.keys())

    for i, line_a in enumerate(lines):
        for line_b in lines[i + 1:]:
            a = rankings_by_line[line_a].sort_values("rank").head(n)["country_iso3"].tolist()
            b = rankings_by_line[line_b].sort_values("rank").head(n)["country_iso3"].tolist()
            overlap = len(set(a) & set(b))
            rows.append({
                "line_a": line_a,
                "line_b": line_b,
                "n": n,
                "overlap": overlap,
                "overlap_pct": overlap / n,
                "top_a": ", ".join(a),
                "top_b": ", ".join(b),
            })

    return pd.DataFrame(rows).sort_values("overlap_pct", ascending=False)

top5_overlap = topn_overlap(line_rankings, n=5)
top10_overlap = topn_overlap(line_rankings, n=10)

display(style_table(
    top5_overlap,
    gradient_cols=["overlap_pct"],
    gradient_cmap="YlOrRd",
    format_dict={"overlap_pct": "{:.0%}"},
).set_caption("Solapamiento Top 5 entre líneas"))

display(style_table(
    top10_overlap,
    gradient_cols=["overlap_pct"],
    gradient_cmap="YlOrRd",
    format_dict={"overlap_pct": "{:.0%}"},
).set_caption("Solapamiento Top 10 entre líneas"))


,line_a,line_b,n,overlap,overlap_pct,top_a,top_b
3,IB,CIB,5,5,100%,"USA, ESP, CAN, CHL, BHS","CAN, USA, ESP, CHL, BHS"
4,PF,AD,5,5,100%,"ESP, CAN, USA, CHL, URY","USA, CAN, ESP, CHL, URY"
0,IB,PF,5,4,80%,"USA, ESP, CAN, CHL, BHS","ESP, CAN, USA, CHL, URY"
1,IB,AD,5,4,80%,"USA, ESP, CAN, CHL, BHS","USA, CAN, ESP, CHL, URY"
2,IB,BD,5,4,80%,"USA, ESP, CAN, CHL, BHS","USA, ESP, CAN, CHL, BRA"
5,PF,BD,5,4,80%,"ESP, CAN, USA, CHL, URY","USA, ESP, CAN, CHL, BRA"
6,PF,CIB,5,4,80%,"ESP, CAN, USA, CHL, URY","CAN, USA, ESP, CHL, BHS"
7,AD,BD,5,4,80%,"USA, CAN, ESP, CHL, URY","USA, ESP, CAN, CHL, BRA"
8,AD,CIB,5,4,80%,"USA, CAN, ESP, CHL, URY","CAN, USA, ESP, CHL, BHS"
9,BD,CIB,5,4,80%,"USA, ESP, CAN, CHL, BRA","CAN, USA, ESP, CHL, BHS"


,line_a,line_b,n,overlap,overlap_pct,top_a,top_b
0,IB,PF,10,8,80%,"USA, ESP, CAN, CHL, BHS, BRA, URY, CRI, PAN, JAM","ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR"
1,IB,AD,10,8,80%,"USA, ESP, CAN, CHL, BHS, BRA, URY, CRI, PAN, JAM","USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG"
2,IB,BD,10,8,80%,"USA, ESP, CAN, CHL, BHS, BRA, URY, CRI, PAN, JAM","USA, ESP, CAN, CHL, BRA, ARG, URY, CRI, MEX, BHS"
3,IB,CIB,10,8,80%,"USA, ESP, CAN, CHL, BHS, BRA, URY, CRI, PAN, JAM","CAN, USA, ESP, CHL, BHS, JAM, BRA, TTO, DOM, URY"
5,PF,BD,10,8,80%,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","USA, ESP, CAN, CHL, BRA, ARG, URY, CRI, MEX, BHS"
7,AD,BD,10,8,80%,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","USA, ESP, CAN, CHL, BRA, ARG, URY, CRI, MEX, BHS"
4,PF,AD,10,7,70%,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG"
6,PF,CIB,10,7,70%,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR","CAN, USA, ESP, CHL, BHS, JAM, BRA, TTO, DOM, URY"
9,BD,CIB,10,7,70%,"USA, ESP, CAN, CHL, BRA, ARG, URY, CRI, MEX, BHS","CAN, USA, ESP, CHL, BHS, JAM, BRA, TTO, DOM, URY"
8,AD,CIB,10,6,60%,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","CAN, USA, ESP, CHL, BHS, JAM, BRA, TTO, DOM, URY"



## 13. Bandas competitivas y gaps: evitar decisiones por diferencias marginales

No toda diferencia ordinal es material. Si dos países tienen scores casi iguales, el ranking debe comunicarse como una banda competitiva. Esto evita sobreinterpretar diferencias que pueden cambiar con pequeñas variaciones de pesos, imputación o universo de países.


In [279]:

# ---------------------------------------------------------------------------
# Bandas y gaps por línea
# ---------------------------------------------------------------------------
def classify_gap(gap: float) -> str:
    """Clasifica la materialidad de un gap de score."""
    if pd.isna(gap):
        return "Sin siguiente país"
    if gap < 0.005:
        return "Empate práctico"
    if gap < 0.015:
        return "Diferencia débil"
    if gap < 0.030:
        return "Diferencia moderada"
    return "Diferencia material"


def assign_tier_by_score(score: float, q80: float, q60: float, q40: float) -> str:
    """Asigna banda de atractividad relativa por cuantiles."""
    if score >= q80:
        return "Alta"
    if score >= q60:
        return "Media-alta"
    if score >= q40:
        return "Media"
    return "Baja"


def strategic_read(tier: str, gap: str) -> str:
    """Genera lectura estratégica combinando banda y gap."""
    if tier == "Alta" and gap == "Diferencia material":
        return "Liderazgo claramente diferenciado"
    if tier == "Alta" and gap in {"Empate práctico", "Diferencia débil"}:
        return "Alta atractividad, pero no distinguible del siguiente"
    if tier == "Media-alta":
        return "Banda competitiva media-alta; requiere tesis específica"
    if tier == "Media":
        return "Atractividad intermedia; priorizar solo si hay racional estratégico"
    return "Baja prioridad relativa en el universo evaluado"

line_tier_tables: Dict[str, pd.DataFrame] = {}

for line, df in line_rankings.items():
    tmp = df.sort_values("score", ascending=False).copy()
    q80, q60, q40 = tmp["score"].quantile([0.80, 0.60, 0.40]).tolist()
    tmp["attractiveness_tier"] = tmp["score"].apply(lambda x: assign_tier_by_score(x, q80, q60, q40))
    tmp["score_gap_next"] = tmp["score"] - tmp["score"].shift(-1)
    tmp["gap_interpretation"] = tmp["score_gap_next"].apply(classify_gap)
    tmp["strategic_read"] = tmp.apply(lambda r: strategic_read(r["attractiveness_tier"], r["gap_interpretation"]), axis=1)
    line_tier_tables[line] = tmp

for line, tmp in line_tier_tables.items():
    display(Markdown(f"### {line} — Bandas y gaps"))
    display(style_table(
        tmp[["country_iso3", "score", "rank", "attractiveness_tier", "score_gap_next", "gap_interpretation", "strategic_read"]].head(15),
        gradient_cols=["score", "score_gap_next"],
        gradient_cmap="RdYlGn",
        format_dict={"score": "{:.3f}", "rank": "{:.0f}", "score_gap_next": "{:.3f}"},
    ))


### IB — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.707,1,Alta,0.024,Diferencia moderada,Baja prioridad relativa en el universo evaluado
1,ESP,0.683,2,Alta,0.012,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
2,CAN,0.671,3,Alta,0.036,Diferencia material,Liderazgo claramente diferenciado
3,CHL,0.635,4,Alta,0.053,Diferencia material,Liderazgo claramente diferenciado
4,BHS,0.582,5,Alta,0.013,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
5,BRA,0.569,6,Media-alta,0.003,Empate práctico,Banda competitiva media-alta; requiere tesis específica
6,URY,0.566,7,Media-alta,0.020,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
7,CRI,0.546,8,Media-alta,0.009,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
8,PAN,0.536,9,Media-alta,0.004,Empate práctico,Banda competitiva media-alta; requiere tesis específica
9,JAM,0.533,10,Media-alta,0.005,Diferencia débil,Banda competitiva media-alta; requiere tesis específica


### PF — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,ESP,0.673,1,Alta,0.032,Diferencia material,Liderazgo claramente diferenciado
1,CAN,0.641,2,Alta,0.015,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
2,USA,0.626,3,Alta,0.029,Diferencia moderada,Baja prioridad relativa en el universo evaluado
3,CHL,0.597,4,Alta,0.042,Diferencia material,Liderazgo claramente diferenciado
4,URY,0.555,5,Alta,0.009,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
5,BRA,0.545,6,Media-alta,0.014,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
6,ARG,0.531,7,Media-alta,0.025,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
7,CRI,0.506,8,Media-alta,0.016,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
8,JAM,0.489,9,Media-alta,0.022,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
9,SUR,0.468,10,Media-alta,0.002,Empate práctico,Banda competitiva media-alta; requiere tesis específica


### AD — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.800,1,Alta,0.057,Diferencia material,Liderazgo claramente diferenciado
1,CAN,0.742,2,Alta,0.091,Diferencia material,Liderazgo claramente diferenciado
2,ESP,0.652,3,Alta,0.015,Diferencia moderada,Baja prioridad relativa en el universo evaluado
3,CHL,0.636,4,Alta,0.019,Diferencia moderada,Baja prioridad relativa en el universo evaluado
4,URY,0.618,5,Alta,0.066,Diferencia material,Liderazgo claramente diferenciado
5,CRI,0.551,6,Media-alta,0.037,Diferencia material,Banda competitiva media-alta; requiere tesis específica
6,BHS,0.514,7,Media-alta,0.040,Diferencia material,Banda competitiva media-alta; requiere tesis específica
7,BLZ,0.474,8,Media-alta,0.005,Empate práctico,Banda competitiva media-alta; requiere tesis específica
8,PAN,0.470,9,Media-alta,0.062,Diferencia material,Banda competitiva media-alta; requiere tesis específica
9,ARG,0.408,10,Media-alta,0.004,Empate práctico,Banda competitiva media-alta; requiere tesis específica


### BD — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.789,1,Alta,0.051,Diferencia material,Liderazgo claramente diferenciado
1,ESP,0.738,2,Alta,0.020,Diferencia moderada,Baja prioridad relativa en el universo evaluado
2,CAN,0.718,3,Alta,0.021,Diferencia moderada,Baja prioridad relativa en el universo evaluado
3,CHL,0.697,4,Alta,0.051,Diferencia material,Liderazgo claramente diferenciado
4,BRA,0.645,5,Alta,0.028,Diferencia moderada,Baja prioridad relativa en el universo evaluado
5,ARG,0.617,6,Media-alta,0.002,Empate práctico,Banda competitiva media-alta; requiere tesis específica
6,URY,0.615,7,Media-alta,0.040,Diferencia material,Banda competitiva media-alta; requiere tesis específica
7,CRI,0.575,8,Media-alta,0.041,Diferencia material,Banda competitiva media-alta; requiere tesis específica
8,MEX,0.534,9,Media-alta,0.009,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
9,BHS,0.526,10,Media-alta,0.001,Empate práctico,Banda competitiva media-alta; requiere tesis específica


### CIB — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,CAN,0.725,1,Alta,0.024,Diferencia moderada,Baja prioridad relativa en el universo evaluado
1,USA,0.701,2,Alta,0.064,Diferencia material,Liderazgo claramente diferenciado
2,ESP,0.636,3,Alta,0.001,Empate práctico,"Alta atractividad, pero no distinguible del siguiente"
3,CHL,0.635,4,Alta,0.089,Diferencia material,Liderazgo claramente diferenciado
4,BHS,0.546,5,Alta,0.025,Diferencia moderada,Baja prioridad relativa en el universo evaluado
5,JAM,0.521,6,Media-alta,0.011,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
6,BRA,0.510,7,Media-alta,0.000,Empate práctico,Banda competitiva media-alta; requiere tesis específica
7,TTO,0.509,8,Media-alta,0.006,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
8,DOM,0.503,9,Media-alta,0.005,Empate práctico,Banda competitiva media-alta; requiere tesis específica
9,URY,0.498,10,Media-alta,0.026,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica



## 14. Dispersión de países entre líneas: mercados multi-tesis vs apuestas selectivas

Un país con posiciones similares en varias líneas es un mercado transversalmente atractivo. Un país con alta dispersión entre líneas no es “malo”; es un mercado cuya atractividad depende de una tesis específica.


In [280]:

# ---------------------------------------------------------------------------
# Dispersión de ranking por país entre líneas
# ---------------------------------------------------------------------------
rank_by_country = pd.DataFrame({
    line: df.set_index("country_iso3")["rank"]
    for line, df in line_rankings.items()
})

rank_dispersion = rank_by_country.copy()
rank_dispersion["rank_mean_across_lines"] = rank_dispersion.mean(axis=1)
rank_dispersion["rank_std_across_lines"] = rank_dispersion.std(axis=1)
rank_dispersion["rank_range_across_lines"] = rank_dispersion.max(axis=1) - rank_dispersion.min(axis=1)
rank_dispersion = rank_dispersion.sort_values("rank_range_across_lines", ascending=False).reset_index().rename(columns={"index": "country_iso3"})

display(style_table(
    rank_dispersion.head(20),
    gradient_cols=["rank_std_across_lines", "rank_range_across_lines"],
    gradient_cmap="YlOrRd",
    format_dict={"rank_mean_across_lines": "{:.1f}", "rank_std_across_lines": "{:.1f}", "rank_range_across_lines": "{:.0f}"},
).set_caption("Países con mayor sensibilidad a la tesis de negocio"))

high_dispersion = rank_dispersion.head(5)["country_iso3"].tolist()
low_dispersion = rank_dispersion.sort_values("rank_range_across_lines").head(5)["country_iso3"].tolist()

display(Markdown(f"""
### Lectura de dispersión

- **Países más dependientes de tesis específica:** {', '.join(high_dispersion)}.  
  Requieren conversación por línea: pueden ser muy atractivos para una tesis y poco atractivos para otra.

- **Países con atractividad transversal más estable:** {', '.join(low_dispersion)}.  
  Son candidatos a discusiones de portafolio regional o multi-línea.
"""))


,country_iso3,IB,PF,AD,BD,CIB,rank_mean_across_lines,rank_std_across_lines,rank_range_across_lines
0,NIC,24,24,24,24,24,24.0,0.0,24
1,GTM,21,23,20,22,23,21.8,1.2,22
2,HND,15,22,22,23,19,20.2,2.9,20
3,BOL,12,21,23,18,16,18.0,3.8,19
4,ECU,18,20,21,19,15,18.6,2.1,19
5,SLV,14,17,19,21,20,18.2,2.5,19
6,MEX,23,19,16,9,12,15.8,5.0,18
7,SUR,22,10,18,17,21,17.6,4.2,18
8,PER,19,15,14,12,13,14.6,2.4,17
9,ARG,20,7,10,6,22,13.0,6.7,16



### Lectura de dispersión

- **Países más dependientes de tesis específica:** NIC, GTM, HND, BOL, ECU.  
  Requieren conversación por línea: pueden ser muy atractivos para una tesis y poco atractivos para otra.

- **Países con atractividad transversal más estable:** USA, ESP, CAN, CHL, URY.  
  Son candidatos a discusiones de portafolio regional o multi-línea.



## 15. Explicabilidad: contribución ponderada + efecto marginal

La contribución ponderada ayuda a explicar el perfil de un país, pero TOPSIS no es lineal. Por eso complementamos con análisis marginal leave-one-variable-out. La combinación permite distinguir:

- **Driver robusto:** alta contribución y efecto marginal positivo.
- **Driver descriptivo:** alta contribución pero bajo efecto marginal.
- **Restricción crítica:** alto shortfall y efecto marginal negativo.
- **Efecto marginal bajo:** variable poco decisiva para el ranking.


In [281]:

# ---------------------------------------------------------------------------
# Explicabilidad: contribuciones y marginales
# ---------------------------------------------------------------------------
contrib_by_line = compute_all_business_line_contributions(
    decision_matrix=decision_matrix,
    weights_audit=weights_audit,
)

marginal_by_line = compute_all_marginal_effects(
    decision_matrix=decision_matrix,
    weights_audit=weights_audit,
    variable_catalog=variable_catalog,
    distance_metric=configs["settings"]["scoring"]["topsis"].get("distance_metric", "euclidean"),
)

explainability_by_line: Dict[str, pd.DataFrame] = {}
for line in line_rankings.keys():
    tmp = combine_contribution_and_marginal(contrib_by_line[line], marginal_by_line[line])
    tmp["driver_class"] = tmp.apply(classify_driver_robustness, axis=1)
    explainability_by_line[line] = tmp

success_box("Explicabilidad calculada", "Se combinaron contribuciones ponderadas y efectos marginales por línea de negocio.")


2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.800 (USA)
2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.800 (USA)
2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.799 (USA)
2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.799 (USA)
2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.800 (USA)
2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.800 (USA)
2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.800 (USA)
2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.800 (USA)
2026-06-25 21:30:06 | INFO     | src.scoring.ranking:rank:133 | 

In [282]:

# ---------------------------------------------------------------------------
# Drivers por línea para el país líder y países prioritarios
# ---------------------------------------------------------------------------
def show_country_drivers(country_iso3: str, line: str, top_n: int = 8) -> None:
    """Muestra drivers y restricciones de un país en una línea específica."""
    display(Markdown(f"### Drivers y restricciones — {country_iso3} en {line}"))
    table = build_country_driver_table(
        explainability_df=explainability_by_line[line],
        country_iso3=country_iso3,
        top_n=top_n,
    )
    display(style_table(
        table,
        gradient_cols=["contribution", "shortfall", "score_effect"],
        gradient_cmap="RdYlGn",
        format_dict={
            "normalized_value": "{:.3f}",
            "contribution": "{:.4f}",
            "shortfall": "{:.4f}",
            "score_effect": "{:.4f}",
            "rank_effect": "{:+.0f}",
        },
    ))

# Mostrar drivers del líder de cada línea.
for line, df in line_rankings.items():
    leader_country = df.sort_values("rank").iloc[0]["country_iso3"]
    show_country_drivers(leader_country, line, top_n=8)


### Drivers y restricciones — USA en IB

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,IB,USA,financial_system_deposits_gdp,financial,0.880,0.090000,0.0792,0.0108,0.0196,+0,Driver,Driver,Driver robusto
1,IB,USA,regulatory_quality,institutional,0.996,0.063000,0.0628,0.0002,0.0131,+0,Driver,Driver,Driver robusto
2,IB,USA,bank_npl_ratio,financial,0.912,0.063000,0.0575,0.0055,0.0103,+1,Driver,Driver,Driver robusto
3,IB,USA,account_ownership,financial,0.982,0.054000,0.0530,0.0010,0.0091,+0,Driver,Driver,Driver descriptivo
4,IB,USA,rule_of_law,institutional,0.838,0.069000,0.0578,0.0112,0.0087,+0,Driver,Driver,Driver descriptivo
5,IB,USA,government_effectiveness,institutional,0.858,0.051000,0.0438,0.0072,0.0052,+0,Driver,Driver,Driver descriptivo
6,IB,USA,bank_concentration_5,financial,1.000,0.036000,0.0360,0.0000,0.0041,+0,Driver,Driver,Driver descriptivo
7,IB,USA,gdp_per_capita_ppp,macro,1.000,0.032000,0.0320,0.0000,0.0032,+0,Driver,Driver,Driver descriptivo
8,IB,USA,bank_capital_rwa,financial,0.189,0.054000,0.0102,0.0438,-0.0401,+0,Restriccion,Restriccion,Restriccion critica
9,IB,USA,interest_rate_spread,financial,0.136,0.036000,0.0049,0.0311,-0.0187,+0,Restriccion,Restriccion,Restriccion critica


### Drivers y restricciones — ESP en PF

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,PF,ESP,digital_payment_adults_pct,digital_tech,0.990,0.188000,0.1861,0.0019,0.1478,+2,Driver,Driver,Driver robusto
1,PF,ESP,account_ownership,financial,1.000,0.055000,0.0550,0.0000,0.0071,+0,Driver,Driver,Driver descriptivo
2,PF,ESP,internet_users_pct,digital_tech,1.000,0.051700,0.0517,0.0000,0.0062,+0,Driver,Driver,Driver descriptivo
3,PF,ESP,inflation_rate,macro,0.986,0.022400,0.0221,0.0003,0.0011,+0,Driver,Driver,Efecto marginal bajo
4,PF,ESP,commercial_bank_branches_per_100k_adults,digital_tech,0.732,0.042300,0.0310,0.0113,0.0010,+0,Driver,Driver,Driver descriptivo
5,PF,ESP,financial_system_deposits_gdp,financial,1.000,0.017500,0.0175,0.0000,0.0007,+0,Driver,Driver,Efecto marginal bajo
6,PF,ESP,domestic_credit_private_gdp,financial,0.968,0.017500,0.0169,0.0006,0.0006,+0,Driver,Driver,Efecto marginal bajo
7,PF,ESP,rule_of_law,institutional,0.828,0.021600,0.0179,0.0037,0.0006,+0,Driver,Driver,Efecto marginal bajo
8,PF,ESP,personal_remittances_gdp,financial,0.086,0.095000,0.0082,0.0868,-0.1045,+0,Restriccion,Restriccion,Restriccion critica
9,PF,ESP,mobile_subscriptions,digital_tech,0.577,0.084600,0.0488,0.0358,-0.0074,+0,Restriccion,Restriccion,Driver descriptivo


### Drivers y restricciones — USA en AD

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,AD,USA,regulatory_quality,institutional,0.996,0.163200,0.1626,0.0006,0.0445,+0,Driver,Driver,Driver robusto
1,AD,USA,secure_internet_servers_per_million,digital_tech,0.921,0.132000,0.1215,0.0105,0.0189,+0,Driver,Driver,Driver robusto
2,AD,USA,rule_of_law,institutional,0.838,0.124800,0.1046,0.0202,0.0063,+0,Driver,Driver,Driver descriptivo
3,AD,USA,internet_users_pct,digital_tech,0.971,0.046200,0.0449,0.0013,0.0025,+0,Driver,Driver,Driver descriptivo
4,AD,USA,control_of_corruption,institutional,0.822,0.086400,0.0710,0.0154,0.0017,+0,Driver,Driver,Driver descriptivo
5,AD,USA,stock_market_cap_gdp,financial,1.000,0.036000,0.0360,0.0000,0.0016,+0,Driver,Driver,Driver descriptivo
6,AD,USA,government_effectiveness,institutional,0.858,0.048000,0.0412,0.0068,0.0012,+0,Driver,Driver,Driver descriptivo
7,AD,USA,digital_payment_adults_pct,digital_tech,0.934,0.026400,0.0247,0.0017,0.0007,+0,Driver,Driver,Efecto marginal bajo
8,AD,USA,ict_goods_exports_pct_total_goods_exports,digital_tech,0.440,0.082500,0.0363,0.0462,-0.0510,+0,Restriccion,Restriccion,Restriccion critica
9,AD,USA,political_stability,institutional,0.389,0.033600,0.0131,0.0205,-0.0083,+0,Restriccion,Restriccion,Driver moderado


### Drivers y restricciones — USA en BD

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,BD,USA,internet_users_pct,digital_tech,0.971,0.102000,0.0991,0.0029,0.0233,+0,Driver,Driver,Driver robusto
1,BD,USA,account_ownership,financial,0.982,0.090000,0.0883,0.0017,0.0179,+0,Driver,Driver,Driver robusto
2,BD,USA,population_total,macro,1.000,0.084000,0.0840,0.0000,0.0160,+0,Driver,Driver,Driver robusto
3,BD,USA,digital_payment_adults_pct,digital_tech,0.934,0.054000,0.0505,0.0035,0.0050,+0,Driver,Driver,Driver descriptivo
4,BD,USA,gdp_per_capita_ppp,macro,1.000,0.042000,0.0420,0.0000,0.0037,+0,Driver,Driver,Driver descriptivo
5,BD,USA,secure_internet_servers_per_million,digital_tech,0.921,0.048000,0.0442,0.0038,0.0037,+0,Driver,Driver,Driver descriptivo
6,BD,USA,gdp_nominal,macro,1.000,0.036000,0.0360,0.0000,0.0027,+0,Driver,Driver,Driver descriptivo
7,BD,USA,inflation_rate,macro,0.985,0.036000,0.0355,0.0005,0.0026,+0,Driver,Driver,Driver descriptivo
8,BD,USA,mobile_subscriptions,digital_tech,0.421,0.066000,0.0278,0.0382,-0.0499,+0,Restriccion,Restriccion,Restriccion critica
9,BD,USA,public_debt_gdp,macro,0.000,0.021000,0.0000,0.0210,-0.0131,+0,Restriccion,Restriccion,Restriccion critica


### Drivers y restricciones — CAN en CIB

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,CIB,CAN,stock_market_cap_gdp,financial,0.925,0.144000,0.1332,0.0108,0.0611,+0,Driver,Driver,Driver robusto
1,CIB,CAN,rule_of_law,institutional,1.000,0.062400,0.0624,0.0000,0.0105,+0,Driver,Driver,Driver robusto
2,CIB,CAN,regulatory_quality,institutional,1.000,0.055200,0.0552,0.0000,0.0081,+0,Driver,Driver,Driver descriptivo
3,CIB,CAN,government_effectiveness,institutional,1.000,0.043200,0.0432,0.0000,0.0049,+0,Driver,Driver,Driver descriptivo
4,CIB,CAN,control_of_corruption,institutional,1.000,0.036000,0.0360,0.0000,0.0033,+0,Driver,Driver,Driver descriptivo
5,CIB,CAN,bank_npl_ratio,financial,1.000,0.032000,0.0320,0.0000,0.0026,+0,Driver,Driver,Driver descriptivo
6,CIB,CAN,inflation_rate,macro,0.987,0.030600,0.0302,0.0004,0.0023,+0,Driver,Driver,Driver descriptivo
7,CIB,CAN,country_risk_premium,institutional,1.000,0.012000,0.0120,0.0000,0.0004,+0,Driver,Driver,Efecto marginal bajo
8,CIB,CAN,bank_capital_rwa,financial,0.169,0.048000,0.0081,0.0399,-0.0315,+0,Restriccion,Restriccion,Restriccion critica
9,CIB,CAN,trade_openness,macro,0.476,0.054400,0.0259,0.0285,-0.0135,+1,Restriccion,Restriccion,Restriccion critica



## 16. Lectura de tesis por línea

Esta sección traduce los rankings a lógica de negocio. El objetivo es que el Comité no vea cinco listas, sino cinco formas de contestar la pregunta: **¿qué significa que un país sea atractivo para esta línea?**


In [283]:

# ---------------------------------------------------------------------------
# Lectura ejecutiva automática por línea
# ---------------------------------------------------------------------------
BUSINESS_THESIS = {
    "IB": "Intermediación Bancaria privilegia profundidad financiera, fondeo, calidad de cartera, solvencia e institucionalidad. Un país alto en IB debe leerse como mercado bancario estructuralmente desarrollable, no necesariamente como mercado digitalmente expansivo.",
    "PF": "Pagos y Flujos privilegia comportamiento transaccional observado, pagos digitales, remesas, apertura y capilaridad operativa. Un país alto en PF indica potencial de mover dinero, recaudar, pagar y conectar flujos.",
    "AD": "Activos Digitales privilegia regulación, estado de derecho, control de corrupción, infraestructura digital segura y sofisticación financiera. Un país alto en AD debe leerse como jurisdicción más apta para innovación digital financiera bajo menor incertidumbre institucional.",
    "BD": "Bancos Digitales privilegia escala retail, población urbana, conectividad, bancarización y potencial de adquisición digital. Un país alto en BD indica potencial de escalar una base masiva de usuarios digitales.",
    "CIB": "Corporate & Investment Banking privilegia escala económica, profundidad de mercado de capitales, apertura, IED, solvencia e institucionalidad. Un país alto en CIB indica mayor potencial para banca corporativa, mercados de capitales y asset management.",
}

for line, thesis in BUSINESS_THESIS.items():
    if line not in line_rankings:
        continue
    df = line_rankings[line].sort_values("rank")
    top = df.head(5)["country_iso3"].tolist()
    display(Markdown(f"""
### {line} — Tesis de atractividad

{thesis}

**Top 5 actual:** {', '.join(top)}.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En {line}, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.
"""))



### IB — Tesis de atractividad

Intermediación Bancaria privilegia profundidad financiera, fondeo, calidad de cartera, solvencia e institucionalidad. Un país alto en IB debe leerse como mercado bancario estructuralmente desarrollable, no necesariamente como mercado digitalmente expansivo.

**Top 5 actual:** USA, ESP, CAN, CHL, BHS.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En IB, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



### PF — Tesis de atractividad

Pagos y Flujos privilegia comportamiento transaccional observado, pagos digitales, remesas, apertura y capilaridad operativa. Un país alto en PF indica potencial de mover dinero, recaudar, pagar y conectar flujos.

**Top 5 actual:** ESP, CAN, USA, CHL, URY.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En PF, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



### AD — Tesis de atractividad

Activos Digitales privilegia regulación, estado de derecho, control de corrupción, infraestructura digital segura y sofisticación financiera. Un país alto en AD debe leerse como jurisdicción más apta para innovación digital financiera bajo menor incertidumbre institucional.

**Top 5 actual:** USA, CAN, ESP, CHL, URY.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En AD, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



### BD — Tesis de atractividad

Bancos Digitales privilegia escala retail, población urbana, conectividad, bancarización y potencial de adquisición digital. Un país alto en BD indica potencial de escalar una base masiva de usuarios digitales.

**Top 5 actual:** USA, ESP, CAN, CHL, BRA.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En BD, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



### CIB — Tesis de atractividad

Corporate & Investment Banking privilegia escala económica, profundidad de mercado de capitales, apertura, IED, solvencia e institucionalidad. Un país alto en CIB indica mayor potencial para banca corporativa, mercados de capitales y asset management.

**Top 5 actual:** CAN, USA, ESP, CHL, BHS.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En CIB, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



## 17. Implicaciones para decisión estratégica

La salida del modelo debe traducirse en decisiones de priorización, no en una carrera por el primer puesto. Las decisiones recomendadas se estructuran en cuatro tipos.


In [284]:

# ---------------------------------------------------------------------------
# Matriz ejecutiva de implicaciones
# ---------------------------------------------------------------------------
def decision_implication(tier: str, gap: str) -> str:
    """Traduce tier y gap en recomendación ejecutiva."""
    if tier == "Alta" and gap == "Diferencia material":
        return "Priorizar análisis de entrada / business case"
    if tier == "Alta":
        return "Mantener en shortlist; diferenciar por drivers y restricciones"
    if tier == "Media-alta":
        return "Profundizar solo si existe encaje con línea específica"
    if tier == "Media":
        return "Monitorear; no priorizar sin tesis adicional"
    return "Descartar del ciclo actual salvo razón estratégica externa"

executive_rows: List[Dict[str, Any]] = []
for line, table in line_tier_tables.items():
    for _, row in table.head(10).iterrows():
        executive_rows.append({
            "business_line": line,
            "country_iso3": row["country_iso3"],
            "rank": row["rank"],
            "score": row["score"],
            "tier": row["attractiveness_tier"],
            "gap": row["gap_interpretation"],
            "decision_implication": decision_implication(row["attractiveness_tier"], row["gap_interpretation"]),
        })

executive_matrix = pd.DataFrame(executive_rows)

display(style_table(
    executive_matrix,
    gradient_cols=["score"],
    gradient_cmap="RdYlGn",
    format_dict={"score": "{:.3f}", "rank": "{:.0f}"},
).set_caption("Matriz ejecutiva de implicaciones por línea"))


,business_line,country_iso3,rank,score,tier,gap,decision_implication
0,IB,USA,1,0.707,Alta,Diferencia moderada,Mantener en shortlist; diferenciar por drivers y restricciones
1,IB,ESP,2,0.683,Alta,Diferencia débil,Mantener en shortlist; diferenciar por drivers y restricciones
2,IB,CAN,3,0.671,Alta,Diferencia material,Priorizar análisis de entrada / business case
3,IB,CHL,4,0.635,Alta,Diferencia material,Priorizar análisis de entrada / business case
4,IB,BHS,5,0.582,Alta,Diferencia débil,Mantener en shortlist; diferenciar por drivers y restricciones
5,IB,BRA,6,0.569,Media-alta,Empate práctico,Profundizar solo si existe encaje con línea específica
6,IB,URY,7,0.566,Media-alta,Diferencia moderada,Profundizar solo si existe encaje con línea específica
7,IB,CRI,8,0.546,Media-alta,Diferencia débil,Profundizar solo si existe encaje con línea específica
8,IB,PAN,9,0.536,Media-alta,Empate práctico,Profundizar solo si existe encaje con línea específica
9,IB,JAM,10,0.533,Media-alta,Diferencia débil,Profundizar solo si existe encaje con línea específica


In [285]:
scores_dir = resolve_data_path(configs["settings"]["data"].get("scores_path", "data/scores"))

# Componentes descompuestos
component_display.to_parquet(scores_dir / "component_decomposition_latest.parquet", index=False)

# TOPSIS by line en formato largo
topsis_long_rows = []
scores_dir = resolve_data_path(configs["settings"]["data"].get("scores_path", "data/scores"))
scores_dir.mkdir(parents=True, exist_ok=True)


for bl_key, bl_df in results["business_line_rankings"].items():
    tmp = ensure_country_column(bl_df)[["country_iso3", "score", "rank"]].copy()
scores_dir = resolve_data_path(configs["settings"]["data"].get("scores_path", "data/scores"))
scores_dir.mkdir(parents=True, exist_ok=True)


topsis_long_rows = []
for bl_key, bl_df in results["business_line_rankings"].items():
    tmp = ensure_country_column(bl_df)[["country_iso3", "score", "rank"]].copy()
    tmp.columns = ["country_iso3", "topsis_score", "topsis_rank"]
    tmp["business_line"] = bl_key
    topsis_long_rows.append(tmp)

topsis_long = pd.concat(topsis_long_rows, ignore_index=True)
topsis_long.to_parquet(scores_dir / "topsis_by_line_latest.parquet", index=False)

radar_by_line_df = results["radar_by_line"].copy()
radar_by_line_df.to_parquet(scores_dir / "radar_by_line_latest.parquet", index=False)

success_box(
    "Persistencia completada",
    f"Archivos guardados en {scores_dir}: component_decomposition_latest, "
    "topsis_by_line_latest, radar_by_line_latest."
)

In [286]:
results["wide_raw"]

variable,gdp_nominal,gdp_per_capita_ppp,gdp_growth,inflation_rate,population_total,urban_population_pct,unemployment_rate,current_account_gdp,public_debt_gdp,trade_openness,...,geographic_distance_km,common_language_spanish,hofstede_pdi,hofstede_idv,hofstede_mas,hofstede_uai,hofstede_lto,hofstede_ivr,colombian_diaspora_stock,cultural_distance_hofstede
country_iso3,,,,,,,,,,,,,,,,,,,,,
ARG,27.182177,30431.193122,-1.342931,219.883929,17.637525,92.274232,7.145,0.893118,68.944038,27.929761,...,8.457655,1.0,49.0,51.0,56.0,86.0,29.0,62.0,9.493336,43.335897
BHS,23.485350,41197.934255,3.378666,0.409162,12.902425,81.324926,9.207,-6.648608,71.457821,79.242459,...,7.779467,0.0,47.0,38.0,65.0,45.0,14.0,67.0,5.963579,45.022217
BLZ,21.887551,14346.518805,3.504664,3.289560,12.941017,41.753211,8.856,-1.612700,105.804729,108.972447,...,7.754053,0.0,80.0,19.0,40.0,86.0,22.5,89.0,3.637586,34.485504
BOL,24.728439,12877.635118,-1.123356,5.099766,16.334280,71.236145,2.967,-2.377848,68.944038,46.977872,...,7.501634,1.0,78.0,23.0,42.0,87.0,21.0,46.0,7.976939,47.791213
BRA,28.413013,22338.476564,3.419315,4.367464,19.172090,87.895855,5.970,-3.027160,81.855443,35.584524,...,8.368925,0.0,69.0,36.0,49.0,76.0,28.0,59.0,10.000206,36.796739
CAN,28.439119,64610.379517,1.554795,2.381584,17.536097,82.700257,6.907,-0.493602,64.887158,65.149921,...,8.571871,0.0,39.0,72.0,52.0,48.0,54.0,68.0,9.938662,79.561297
CHL,26.523168,36181.156617,2.644312,4.297639,16.799412,88.995091,8.974,-1.469331,68.944038,63.859857,...,8.354910,1.0,63.0,49.0,28.0,86.0,12.0,68.0,10.581166,44.821870
CRI,25.280825,31106.764356,4.321224,-0.412853,15.450599,79.310767,6.843,-0.946456,105.804729,71.331014,...,7.074117,1.0,35.0,15.0,21.0,86.0,22.5,89.0,9.142276,58.423026
DOM,25.545821,27541.662528,4.953522,3.302233,16.251538,72.412187,5.092,-3.353013,84.662111,51.783514,...,7.082549,1.0,65.0,38.0,65.0,45.0,11.0,54.0,10.320651,46.658333


In [287]:
fig = px.bar(
    results["wide_raw"]["gdp_nominal"].reset_index().sort_values("gdp_nominal"),
    x="gdp_nominal",
    y="country_iso3",
    orientation="h",
    color="gdp_nominal",
    color_continuous_scale=[[0, CIBEST["green"]], [0.5, CIBEST["gold"]], [1, CIBEST["red"]]],
    title="Países con mayor GDP nominal",
)
fig.update_layout(xaxis_title="GDP nominal", yaxis_title="País")
fig.show()

In [288]:
# Descubrir variables disponibles y mapearlas a dimensiones
available_vars = [c for c in wide_raw.columns if c != "country_iso3"]
print(f"\n{len(available_vars)} variables disponibles en wide_raw:")

dim_vars_available: Dict[str, List[str]] = {}
for var in available_vars:
    meta = variable_catalog.get(var, {})
    dim = meta.get("dimension", "other")
    dim_vars_available.setdefault(dim, []).append(var)
    
for dim, vars_list in sorted(dim_vars_available.items()):
    print(f"  {dim}: {', '.join(vars_list)}")


46 variables disponibles en wide_raw:
  digital_tech: internet_users_pct, mobile_subscriptions, digital_payment_adults_pct, secure_internet_servers_per_million, commercial_bank_branches_per_100k_adults, atms_per_100k_adults, ict_goods_exports_pct_total_goods_exports
  financial: domestic_credit_private_gdp, account_ownership, interest_rate_spread, bank_npl_ratio, stock_market_cap_gdp, personal_remittances_gdp, bank_concentration_5, financial_system_deposits_gdp, bank_capital_rwa
  institutional: regulatory_quality, government_effectiveness, rule_of_law, political_stability, voice_accountability, control_of_corruption, country_risk_premium, heritage_efi
  macro: gdp_nominal, gdp_per_capita_ppp, gdp_growth, inflation_rate, population_total, urban_population_pct, unemployment_rate, current_account_gdp, public_debt_gdp, trade_openness, fdi_net_inflows_gdp, weighted_mean_applied_tariff_all_products
  other: cultural_distance_hofstede
  proximity: geographic_distance_km, common_language_spa

In [289]:
# Helper genérico para barplot horizontal de una variable
def barplot_variable(var_name: str, title: str, xlabel: str, reverse_color: bool = False):
    if var_name not in wide_raw.columns:
        display(Markdown(f"*Variable `{var_name}` no disponible en wide_raw.*"))
    return
    plot_df = results["wide_raw"][results["wide_raw"]["country_iso3"].isin(top_15)][["country_iso3", var_name]].dropna()
    plot_df = plot_df.sort_values(var_name, ascending=True)
    cscale = [[0, C["green"]], [0.5, C["gold"]], [1, C["red"]]] if reverse_color else [[0, C["gray_light"]], [1, C["gold"]]]
    fig = px.bar(plot_df, x=var_name, y="country_iso3", orientation="h",
                title=title, color=var_name, color_continuous_scale=cscale)
    fig.update_layout(xaxis_title=xlabel, yaxis_title="", height=480, font=dict(family="Arial"))
    fig.show()

In [290]:
for var in dim_vars_available.get("macro", []):
    print(var)
    barplot_variable(var, f"{var} por país", var, reverse_color=True)

gdp_nominal
gdp_per_capita_ppp
gdp_growth
inflation_rate
population_total
urban_population_pct
unemployment_rate
current_account_gdp
public_debt_gdp
trade_openness
fdi_net_inflows_gdp
weighted_mean_applied_tariff_all_products


In [291]:
display(Markdown("### D1 — Macroeconómica"))
for var in dim_vars_available.get("macro", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D1 — Macroeconómica


## 18. Limitaciones interpretativas que deben explicitarse

1. **TOPSIS es relativo al universo evaluado.** Si se excluyen o agregan países, cambian ideales positivos/negativos y pueden cambiar scores.
2. **Los pesos expresan tesis, no verdades universales.** Una línea puede tener una tesis bien estructurada y aun así correlacionar con otra si los países comparten fundamentos.
3. **Las contribuciones ponderadas no son una descomposición exacta del score TOPSIS.** Sirven para narrativa y perfilamiento; el efecto marginal ayuda a validar materialidad.
4. **IPC y Trend modulan el ranking, pero no sustituyen fundamentos estructurales.** Un país cercano no debe priorizarse si sus restricciones estructurales son severas.
5. **La imputación debe ser residual.** La exclusión por calidad debe ocurrir antes de imputar para no construir ideales con países artificialmente completados.



## 19. Conclusiones ejecutivas del Notebook 03

1. **RADAR global prioriza mercados que combinan fundamentos, cercanía y momentum.** No es equivalente a TOPSIS puro.
2. **Los países estructuralmente fuertes pueden bajar en RADAR si tienen menor afinidad con Colombia o bajo momentum reciente.** Esto no es un error: es parte de la lógica estratégica del score compuesto.
3. **Los rankings por línea reflejan tesis de negocio, no listas independientes.** La correlación alta entre líneas digitales puede ser válida cuando existe un factor común de madurez digital-institucional.
4. **La decisión debe comunicarse por bandas y gaps.** Diferencias marginales no deben convertirse en narrativas de superioridad estratégica.
5. **La explicabilidad requiere cruzar contribución y marginal.** Los drivers robustos son los que combinan peso, desempeño y efecto en ranking.
6. **El siguiente paso obligatorio es robustez probabilística.** La versión determinística debe validarse en Notebook 04 mediante Monte Carlo RADAR.

---

## Síntesis Ejecutiva

RADAR Cibest no entrega un ranking único; entrega un mapa de tesis de atractividad por país y línea. La lectura robusta combina ranking, banda, gap, componentes RADAR, correlación entre líneas, drivers, restricciones y sensibilidad a la tesis de negocio. La decisión ejecutiva debe concentrarse en mercados con alta atractividad, brecha material, drivers robustos y factibilidad estratégica para Cibest.
